# Phase 4: UP³RE-Net Research Training

**Uncertainty-Propagated Per-Pixel Reweighting Ensemble Network**

Novel method: Bagging ensemble → per-pixel uncertainty maps → UWACL-guided boosting → confidence-aware inference

Patent claims 1–6 documented in `docs/RESEARCH_NOVELTY.md`

In [ ]:
# Cell 0: Setup
import os, sys, json, pickle, random, warnings
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn

# Ensure src/ is on path
root = Path.cwd()
while not (root / 'src').exists():
    root = root.parent
sys.path.insert(0, str(root))
os.chdir(root)

# Install timm if needed
try:
    import timm
except ImportError:
    !pip install timm -q

# Disable HF symlinks warning
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
warnings.filterwarnings('ignore')

# Project imports
from src import *
from src.data_loader import DatasetConfig, DataPathManager, VolumeWiseSplitter, create_2d_dataloaders
from src.preprocessing import PreprocessingTransform, AugmentedPreprocessingTransform, CLAHEProcessor
from src.models import create_model, EnsembleWrapper, count_params
from src.losses import CombinedLoss, UncertaintyWeightedLoss
from src.metrics import dice_coefficient, iou_score, ensemble_uncertainty, calibration_error
from src.trainer import Trainer, UncertaintyPrecomputer, UncertaintyGuidedTrainer, evaluate_model

# Device
device = setup_device()
set_seed(42)
cfg = PHASE4_RESEARCH_CONFIG

print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Config: {cfg['num_bagging']} bagging members, {cfg['epochs_stage1']}+{cfg['epochs_stage2']} epochs")




---
## Cell 1: Data Preparation
Load volume index, create 3 stratified bootstrap DataLoaders for bagging, and standard val/test DataLoaders.

In [ ]:
# Cell 1: Data Preparation

# Load cached volume index
index_path = DatasetConfig.PREP_OUTPUT_DIR / 'volume_index.pkl'
if not index_path.exists():
    print('Rebuilding volume index...')
    volume_index = DataPathManager().build_index()
else:
    with open(index_path, 'rb') as f:
        volume_index = pickle.load(f)
    print(f'Loaded volume_index.pkl: {len(volume_index["volumes"])} volumes, {len(volume_index["image_paths"])} slices')

# Load splits
splitter = VolumeWiseSplitter()
splits = splitter.load_splits(DatasetConfig.SPLITS_DIR)
train_ids, val_ids, test_ids = splits['train'], splits['val'], splits['test']
print(f'Train: {len(train_ids)} vols, Val: {len(val_ids)} vols, Test: {len(test_ids)} vols')

from PIL import Image
# Identify tumor-positive volumes (for stratified bootstrap)
# Sample slices evenly across each volume — first 5 slices are always empty
tumor_vols = []
for vid in train_ids:
    vol_masks = volume_index['mask_paths'].get(vid, [])
    n_check = min(10, len(vol_masks))
    indices = np.linspace(0, len(vol_masks)-1, n_check, dtype=int)
    sample = [vol_masks[i] for i in indices]
    has_tumor = any(np.array(Image.open(p)).max() > 0 for p in sample)
    if has_tumor:
        tumor_vols.append(vid)
if not tumor_vols:
    print('[WARNING] No tumor-positive volumes detected. Using all train IDs.')
    tumor_vols = train_ids.copy()
print(f'Tumor-positive training volumes: {len(tumor_vols)}/{len(train_ids)}')

# Transforms
clahe = CLAHEProcessor(clip=2.0, grid=(8, 8))
train_transform = AugmentedPreprocessingTransform((256, 256), -100, 400, clahe)
val_transform = PreprocessingTransform((256, 256), -100, 400)

# Create 3 stratified bootstrap train loaders
train_loaders = []
for i in range(cfg['num_bagging']):
    sample = random.choices(train_ids, k=len(train_ids))
    if not any(v in tumor_vols for v in sample):
        sample[random.randint(0, len(sample)-1)] = random.choice(tumor_vols)
    loader, _, _ = create_2d_dataloaders(
        volume_index, sample, [], [],
        batch_size=cfg['batch_size'], num_workers=cfg['num_workers'],
        pin_memory=cfg['pin_memory'],
        transform_train=train_transform,
        transform_val=val_transform,
    )
    train_loaders.append(loader)
    print(f'  Bootstrap {i+1}: {len(sample)} vols, unique={len(set(sample))}, tumor={sum(v in tumor_vols for v in sample)}')

# Standard val loader
_, val_loader, _ = create_2d_dataloaders(
    volume_index, [], val_ids, [],
    batch_size=cfg['batch_size'], num_workers=cfg['num_workers'],
    pin_memory=cfg['pin_memory'],
    transform_val=val_transform,
)

# Standard test loader
_, _, test_loader = create_2d_dataloaders(
    volume_index, [], [], test_ids,
    batch_size=cfg['batch_size'], num_workers=cfg['num_workers'],
    pin_memory=cfg['pin_memory'],
    transform_val=val_transform,
)
print(f'Created {len(train_loaders)} train loaders, 1 val loader, 1 test loader')


\n\n



---
## Cell 2: Stage 1 — Bagging Ensemble
Train 3 MobileNetV2U-Net models on 3 stratified bootstrap samples.

*Patent Claim 1: Training a plurality of base segmentation models on bootstrap samples.*

In [ ]:
# Cell 2: Stage 1 — Bagging Ensemble
import time

os.makedirs(cfg['models_dir'], exist_ok=True)
ensemble_members = []
stage1_results = []

for i, loader in enumerate(train_loaders):
    print(f'\n{"="*60}')
    print(f'Training ensemble member {i+1}/{cfg["num_bagging"]}...')
    
    model = create_model('mobilenetv2_unet', pretrained=True).to(device)
    params = count_params(model)
    print(f'  Model params: {params:,}')
    
    trainer = Trainer(model, loader, val_loader, cfg)
    t0 = time.time()
    best_model = trainer.fit(epochs=cfg['epochs_stage1'])
    elapsed = (time.time() - t0) / 3600
    
    torch.save(best_model.state_dict(), f"{cfg['models_dir']}/member_{i}.pth")
    ensemble_members.append(best_model)
    stage1_results.append({
        'member': i,
        'best_val_dice': trainer.best_val_dice,
        'hours': elapsed,
        'epochs_trained': len(trainer.history['train_loss']),
    })
    print(f'  Member {i+1} done. Best val Dice: {trainer.best_val_dice:.4f}, Time: {elapsed:.1f} hrs')
    print(f'  History: last train loss={trainer.history["train_loss"][-1]:.4f}')

print(f'\n{"="*60}')
print('Stage 1 complete — 3 ensemble members trained and saved.')

---
## Cell 3: Uncertainty Pre-computation
Run all 3 ensemble members on full training set sequentially, compute per-pixel variance.
Save batch-level .pt files.

*Patent Claim 1 continued: Generating per-pixel uncertainty maps from ensemble variance.*

In [ ]:
# Cell 3: Uncertainty Pre-computation
import shutil

# Clean and recreate uncertainty directory
unc_dir = Path(cfg['uncertainty_dir'])
if unc_dir.exists():
    shutil.rmtree(unc_dir)
unc_dir.mkdir(parents=True)

print('Computing per-pixel uncertainty on training set...')
t0 = time.time()

# Move ensemble members to CPU to prep for sequential loading
for m in ensemble_members:
    m.to('cpu')
torch.cuda.empty_cache()

precomputer = UncertaintyPrecomputer(ensemble_members, train_loaders[0], device, output_dir=unc_dir)
num_batches = precomputer.run()

elapsed = time.time() - t0
total_mb = sum(f.stat().st_size for f in unc_dir.glob('*.pt')) / 1e6
print(f'Pre-computed uncertainty for {num_batches} batches ({total_mb:.0f} MB) in {elapsed/60:.1f} min')
print(f'Files saved to: {unc_dir}')

---
## Cell 4: Stage 2 — Uncertainty-Guided Boosting with UWACL
Train 4th MobileNetV2U-Net with uncertainty-weighted loss.
Uses pre-computed uncertainty maps from Cell 3.

*Patent Claims 2–4: UWACL loss, per-pixel weight formula, dynamic tau scheduling.*

In [ ]:
# Cell 4: Stage 2 — Uncertainty-Guided Boosting
from torch.utils.data import Dataset, DataLoader

# Wrap training dataset with pre-computed uncertainty
class UncertaintyWeightedDataset(Dataset):
    def __init__(self, base_loader, uncertainty_dir):
        self.base_dataset = base_loader.dataset
        self.uncertainty_dir = Path(uncertainty_dir)
        self.batch_size = base_loader.batch_size
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        item = self.base_dataset[idx]
        batch_idx = idx // self.batch_size
        u_path = self.uncertainty_dir / f'batch_{batch_idx:05d}.pt'
        if u_path.exists():
            u_data = torch.load(u_path)
            pos_in_batch = idx % self.batch_size
            if pos_in_batch < u_data['uncertainty'].size(0):
                item['uncertainty'] = u_data['uncertainty'][pos_in_batch]
        return item

uw_dataset = UncertaintyWeightedDataset(train_loaders[0], unc_dir)
uw_loader = DataLoader(
    uw_dataset, batch_size=cfg['batch_size'], shuffle=True,
    num_workers=2, pin_memory=cfg['pin_memory'],
)

print('Training Stage 2: Uncertainty-guided boosting with UWACL...')
model_stage2 = create_model('mobilenetv2_unet', pretrained=True).to(device)
print(f'  Model params: {count_params(model_stage2):,}')

# Dynamically reference the loss from trainer (UncertaintyWeightedLoss with beta=5, tau=0.1)
trainer_s2 = UncertaintyGuidedTrainer(model_stage2, uw_loader, val_loader, cfg)

t0 = time.time()
best_boosted = trainer_s2.fit(epochs=cfg['epochs_stage2'])
elapsed = (time.time() - t0) / 3600

torch.save(best_boosted.state_dict(), f"{cfg['models_dir']}/boosted.pth")
print(f'Stage 2 done. Best val Dice: {trainer_s2.best_val_dice:.4f}, Time: {elapsed:.1f} hrs')
print(f'History: last train loss={trainer_s2.history["train_loss"][-1]:.4f}')

---
## Cell 5: Evaluation
Compare Single Model vs Bagging Ensemble vs UP³RE Full on test set.

In [ ]:
# Cell 5: Evaluation

results = {}

# 1. Single model (member 0)
print('Evaluating single model (member 0)...')
m0 = ensemble_members[0].to(device).eval()
r0 = evaluate_model(m0, test_loader, device, return_uncertainty=False)
results['Single (member 0)'] = r0
print(f'  Dice: {r0["dice"]:.4f}, IoU: {r0["iou"]:.4f}, ECE: {r0["ece"]:.4f}')
m0.to('cpu')

# 2. Bagging ensemble (3 members)
print('Evaluating bagging ensemble...')
bagging_ensemble = EnsembleWrapper(ensemble_members)
r_bag = evaluate_model(bagging_ensemble, test_loader, device, return_uncertainty=True)
results['Bagging Ensemble'] = r_bag
print(f'  Dice: {r_bag["dice"]:.4f}, IoU: {r_bag["iou"]:.4f}, ECE: {r_bag["ece"]:.4f}, '
      f'Unc: {r_bag.get("uncertainty_mean", 0):.6f}')

# 3. UP³RE Full (3 bagging + 1 boosted)
print('Evaluating UP³RE Full...')
all_models = ensemble_members + [best_boosted]
upre_ensemble = EnsembleWrapper(all_models)
r_full = evaluate_model(upre_ensemble, test_loader, device, return_uncertainty=True)
results['UP³RE Full'] = r_full
print(f'  Dice: {r_full["dice"]:.4f}, IoU: {r_full["iou"]:.4f}, ECE: {r_full["ece"]:.4f}, '
      f'Unc: {r_full.get("uncertainty_mean", 0):.6f}')

# Results table
print(f'\n{"="*70}')
print(f'{"Model":<25} {"Dice":>8} {"IoU":>8} {"ECE":>8} {"Uncertainty":>12}')
print(f'{"-"*70}')
for name, r in results.items():
    unc = r.get('uncertainty_mean', 0)
    print(f'{name:<25} {r["dice"]:>8.4f} {r["iou"]:>8.4f} {r["ece"]:>8.4f} {unc:>12.6f}')

# Save results
os.makedirs(cfg['outputs_dir'], exist_ok=True)
with open(f"{cfg['outputs_dir']}/comparison_results.json", 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f'Results saved to {cfg["outputs_dir"]}/comparison_results.json')

---
## Cell 6: Ablation Study
Compare UWACL (boosted model) vs standard CombinedLoss (member 1) — same architecture, different loss.

In [ ]:
# Cell 6: Ablation — UWACL vs CombinedLoss

# Member 1 was trained with CombinedLoss (standard)
# Boosted model was trained with UWACL

ablation = {}

print('Ablation: UWACL vs CombinedLoss')
print(f'{"-"*60}')

for label, model in [('CombinedLoss (member 1)', ensemble_members[1]), ('UWACL (boosted)', best_boosted)]:
    r = evaluate_model(model.to(device).eval(), test_loader, device)
    model.to('cpu')
    ablation[label] = r
    print(f'{label:<30} Dice: {r["dice"]:.4f}, IoU: {r["iou"]:.4f}, ECE: {r["ece"]:.4f}')

gain = {
    'dice_gain': ablation['UWACL (boosted)']['dice'] - ablation['CombinedLoss (member 1)']['dice'],
    'iou_gain': ablation['UWACL (boosted)']['iou'] - ablation['CombinedLoss (member 1)']['iou'],
    'ece_reduction': ablation['CombinedLoss (member 1)']['ece'] - ablation['UWACL (boosted)']['ece'],
}
print(f'\nUWACL improvement vs CombinedLoss:')
print(f'  Dice gain:     {gain["dice_gain"]:+.4f}')
print(f'  IoU gain:      {gain["iou_gain"]:+.4f}')
print(f'  ECE reduction: {gain["ece_reduction"]:+.4f}')

with open(f"{cfg['outputs_dir']}/ablation_results.json", 'w') as f:
    json.dump({'ablation': ablation, 'gain': gain}, f, indent=2, default=str)
print(f'Ablation results saved.')

---
## Cell 7: Figures
Generate comparison bar chart, uncertainty maps, and calibration plot.

In [ ]:
# Cell 7: Figures
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig_dir = Path(f"{cfg['outputs_dir']}/figures")
fig_dir.mkdir(parents=True, exist_ok=True)

# Figure 1: Comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
names = list(results.keys())
metrics_to_plot = ['dice', 'iou', 'ece']
titles = ['Dice Coefficient', 'IoU Score', 'Calibration Error (ECE)']
colors = ['#4CAF50', '#2196F3', '#FF9800', '#9C27B0']

for ax, metric, title in zip(axes, metrics_to_plot, titles):
    values = [results[n][metric] for n in names]
    bars = ax.bar(names, values, color=colors[:len(names)])
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xticklabels(names, rotation=15, ha='right')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(fig_dir / 'comparison_bar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 saved: comparison_bar_chart.png')

# Figure 2: Uncertainty maps on 6 test examples
fig, axes = plt.subplots(6, 4, figsize=(16, 24))
count = 0
with torch.no_grad():
    for batch in test_loader:
        images = batch['image'].to(device)
        masks = batch['mask']
        mean_pred, variance = upre_ensemble.predict_with_uncertainty(images, device)
        for i in range(images.size(0)):
            if count >= 6:
                break
            img = images[i, 0].cpu().numpy()
            gt = masks[i, 0].numpy()
            pred = mean_pred[i, 0].cpu().numpy()
            unc = variance[i, 0].cpu().numpy()
            axes[count, 0].imshow(img, cmap='gray')
            axes[count, 0].set_title('Input')
            axes[count, 0].axis('off')
            axes[count, 1].imshow(gt, cmap='jet', vmin=0, vmax=1)
            axes[count, 1].set_title('Ground Truth')
            axes[count, 1].axis('off')
            axes[count, 2].imshow(pred, cmap='jet', vmin=0, vmax=1)
            axes[count, 2].set_title('Prediction')
            axes[count, 2].axis('off')
            axes[count, 3].imshow(unc, cmap='hot', vmin=0, vmax=0.25)
            axes[count, 3].set_title('Uncertainty')
            axes[count, 3].axis('off')
            count += 1
        if count >= 6:
            break

plt.tight_layout()
plt.savefig(fig_dir / 'uncertainty_maps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 2 saved: uncertainty_maps.png')

# Figure 3: Calibration plot
fig, ax = plt.subplots(figsize=(8, 8))
n_bins = 15
for name, model_for_eval in [('Single', m0.to(device).eval()),
                              ('UP\u00b3RE', upre_ensemble)]:
    all_probs, all_targets = [], []
    with torch.no_grad():
        for batch in test_loader:
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            if hasattr(model_for_eval, 'predict_with_uncertainty'):
                pred, _ = model_for_eval.predict_with_uncertainty(images, device)
            else:
                pred = torch.sigmoid(model_for_eval(images))
            all_probs.append(pred.cpu())
            all_targets.append(masks.cpu())
    m0.to('cpu')
    probs = torch.cat(all_probs).view(-1)
    targets = torch.cat(all_targets).view(-1)
    accuracies = (probs > 0.5).float().eq(targets).float()
    bins = torch.linspace(0, 1, n_bins + 1)
    bin_accs, bin_confs = [], []
    for i in range(n_bins):
        mask = (probs > bins[i]) & (probs <= bins[i+1])
        if mask.sum() > 0:
            bin_accs.append(accuracies[mask].mean().item())
            bin_confs.append(probs[mask].mean().item())
    ax.plot(bin_confs, bin_accs, marker='o', label=name, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
ax.set_xlabel('Confidence', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Calibration Plot', fontsize=14, fontweight='bold')
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(fig_dir / 'calibration_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 3 saved: calibration_plot.png')

print(f'All figures saved to {fig_dir}')

---
## Cell 8: Export + Patent Draft
Save final models and generate patent draft document.

In [ ]:
# Cell 8: Export + Patent Draft

# Save results summary
summary = {
    'stage1': stage1_results,
    'comparison': {k: {mk: v for mk, v in r.items() if isinstance(v, (int, float))}
                   for k, r in results.items()},
    'ablation': gain,
    'config': {k: v for k, v in cfg.items() if not isinstance(v, (Path, dict))},
    'params_count': count_params(ensemble_members[0]),
}
with open(f"{cfg['outputs_dir']}/research_summary.json", 'w') as f:
    json.dump(summary, f, indent=2, default=str)

# Generate patent draft markdown
patent_md = f"""# Uncertainty-Propagated Per-Pixel Reweighting Ensemble Network (UP\u00b3RE-Net)

## Patent Application (Provisional Draft)

### Background
Liver tumor segmentation from CT scans faces extreme class imbalance (937:1) and high uncertainty
at tumor boundaries. Existing ensemble methods average predictions without modeling per-pixel
uncertainty; existing uncertainty methods don't use uncertainty to guide training.

### Summary of the Invention
UP\u00b3RE-Net addresses both limitations by (1) training a bagging ensemble to capture epistemic
uncertainty via prediction variance, (2) propagating per-pixel uncertainty as loss weights to a
boosting stage, and (3) outputting both segmentation and confidence maps.

### Claims

**Claim 1:** A computer-implemented method comprising:
(a) Training base segmentation models on bootstrap samples to form a bagging ensemble;
(b) Generating per-pixel uncertainty maps from ensemble prediction variance;
(c) Computing per-pixel weight maps from said uncertainty maps;
(d) Training a boosting model with a loss function weighted by said weight maps;
(e) Combining all models to produce a final segmentation.

**Claim 2:** The method of claim 1, wherein the per-pixel weight is w = 1 + beta * (1 - exp(-variance / tau)).

**Claim 3:** The method of claim 1, wherein the loss function is UWACL (Uncertainty-Weighted Adaptive
Compound Loss) combining weighted BCE and Dice per-pixel.

**Claim 4:** The method of claim 2, wherein tau is dynamically reduced according to a schedule.

**Claim 5:** The method of claim 1, further comprising a consistency regularization term between
ensemble members.

**Claim 6:** A system implementing the method of claim 1, outputting both segmentation mask and
per-pixel confidence map.

### Experimental Results (RTX 3050 Ti, 4GB)

| Method | Dice | IoU | ECE | Params |
|--------|------|-----|-----|--------|
| Single model | {results['Single (member 0)']['dice']:.4f} | {results['Single (member 0)']['iou']:.4f} | {results['Single (member 0)']['ece']:.4f} | {count_params(ensemble_members[0]):,} |
| Bagging ensemble | {results['Bagging Ensemble']['dice']:.4f} | {results['Bagging Ensemble']['iou']:.4f} | {results['Bagging Ensemble']['ece']:.4f} | {count_params(ensemble_members[0]) * 3:,} |
| UP\u00b3RE Full | {results['UP\u00b3RE Full']['dice']:.4f} | {results['UP\u00b3RE Full']['iou']:.4f} | {results['UP\u00b3RE Full']['ece']:.4f} | {count_params(ensemble_members[0]) * 4:,} |

Ablation: UWACL improves Dice by {gain.get('dice_gain', 0):+.4f} over CombinedLoss.
"""

patent_path = f"{cfg['outputs_dir']}/patent_draft.md"
with open(patent_path, 'w') as f:
    f.write(patent_md)
print(f'Patent draft saved to {patent_path}')

print(f'\n{"="*60}')
print('UP\u00b3RE-Net research complete!')
print(f'Models saved to: {cfg["models_dir"]}/')
print(f'Results saved to: {cfg["outputs_dir"]}/')
print(f'Patent draft: {patent_path}')
print(f'For patent review, see: docs/RESEARCH_NOVELTY.md')